# Ingest raw CSV files into S3 and discover schema with a Glue crawler

Three payment processors dump CSV exports into a shared inbox every morning. Nobody catalogs them. Nobody checks the schemas. The analytics team at a Series A fintech startup has been copy-pasting column names into spreadsheets by hand for six months, and last Friday they shipped a dashboard with the wrong column mapped to `amount_cents` because processor_gamma uses epoch-second timestamps where the other two use ISO 8601. The VP of Analytics has mass-pinged the data engineering channel: "Fix this before Monday standup or I am buying a Fivetran contract."

Your shift starts now. You have one deliverable: stand up a staging bucket, drop in three sample CSVs (one per processor, each with its own quirks), run a Glue crawler to discover the schema automatically, verify the Data Catalog got the columns right, and confirm the row count from Athena matches the source files. When you are done, the analytics team has a cataloged table they can query without guessing column names, and you have a cleanup script that leaves zero resources behind.

The three CSVs are deliberately messy:
- **processor_alpha.csv**: compact ISO 8601 timestamps with no separators (20241101T000000Z instead of 2024-11-01T00:00:00Z).
- **processor_beta.csv**: clean headers, ISO 8601 timestamps in `created_at`.
- **processor_gamma.csv**: clean headers, but `created_at` is epoch seconds.

The crawler's behavior on these imperfections is where the real learning happens.

**Time.** Plan for about 45 minutes hands-on. If you hit snags and need to debug, budget up to 90 minutes. The cleanup cell at the bottom keeps your AWS costs near zero either way.


## What this lab costs

This lab costs about 3 cents if everything goes perfectly on the first try. It will not go perfectly on the first try (that is the point of doing labs). A realistic session with one or two crawler re-runs after fixing the IAM policy lands around 10 to 15 cents.

Here is where the money goes:

- **Glue crawler run**: about 1.5 cents each (2 DPU minimum, 1 minute minimum, \$0.44 per DPU-hour). You will probably run the crawler 2 or 3 times while debugging permissions.
- **Athena SELECT COUNT(*)**: a fraction of a penny per query (\$5 per TB scanned, and your sample CSVs are under 5 MB).
- **S3 storage**: the sample CSVs total under 5 MB, covered by the always-free tier.
- **IAM role, Glue database, CloudWatch logs**: all free tier.

After every checkpoint, the notebook prints a wallet-check line so you can watch the running total. The cleanup cell at the end tears down every resource so your bill stops the moment you finish. Check your AWS Billing console in 24 hours to see the exact number.


## Learning outcomes

By the end of this lab, you will have demonstrated that you can:

- Provision an S3 bucket scoped to a single lab session and tag every resource with `labrun:lab-slug` so cleanup is verifiable.
- Upload CSV files with deliberate schema inconsistencies (missing headers, mixed timestamp formats, trailing delimiters) and observe how a Glue crawler handles each case.
- Configure an IAM role with a least-privilege inline policy that lets a Glue crawler list and read objects under a single S3 prefix.
- Run a Glue crawler, inspect the resulting Data Catalog table, and confirm the schema matches the source CSVs.
- Query the catalog table from Athena and assert that the row count matches the source files.
- Tear down every resource with the three-phase cleanup protocol so your AWS account returns to zero ongoing charges from this lab.

These map to AWS DEA-C01 Domain 1: Data Ingestion and Transformation (34% of exam weight).


In [ ]:
# NBVAL_SKIP
# Setup: install labrun-checks, prompt for AWS credentials, register the session.
# This cell is NBVAL_SKIP because it requires a live session token and live AWS creds.

!pip install --quiet labrun-checks==0.3.0

import getpass

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "s3-raw-ingestion"

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
REGION = input("AWS region (default us-east-1): ").strip() or "us-east-1"

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
# register() returns None. Do not assign its return value.

print("Session registered for lab_id:", LAB_ID)
print("AWS region in use:", REGION)


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# This cell is NBVAL_SKIP because it scans live AWS for tagged resources.

import atexit
import time

import boto3
from botocore.exceptions import ClientError

LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[0].slug exactly

# Resource names follow a deterministic pattern so the cleanup manifest can name
# them up front. Each resource is tagged with labrun:lab-slug=s3-raw-ingestion
# so the Phase 3 tag audit can confirm zero orphans remain after cleanup.
account_id = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    region_name=REGION,
).get_caller_identity()["Account"]

BUCKET_NAME = f"labrun-{LAB_ID}-{account_id}"
GLUE_DATABASE = f"labrun_{LAB_ID.replace('-', '_')}"
GLUE_CRAWLER = f"labrun-{LAB_ID}-crawler"
GLUE_ROLE_NAME = f"labrun-{LAB_ID}-glue-role"

# Critical-first ordering. None of Lab 1's resources are hourly-billed (no
# Redshift, no DMS, no Kinesis), so the order here is reverse-creation. Future
# labs that create critical resources put them at the top of the manifest.
# IAM role is freed after the crawler that uses it.
CLEANUP_MANIFEST = [
    CleanupResource(
        type="glue_crawler",
        id=GLUE_CRAWLER,
        region=REGION,
        cli_delete_command=f"aws glue delete-crawler --name {GLUE_CRAWLER}",
    ),
    CleanupResource(
        type="iam_role",
        id=GLUE_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {GLUE_ROLE_NAME}",
    ),
    CleanupResource(
        type="glue_database",
        id=GLUE_DATABASE,
        region=REGION,
        cli_delete_command=f"aws glue delete-database --name {GLUE_DATABASE}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    Runs run_cleanup against the manifest. Errors are swallowed because
    atexit handlers must not raise; the dedicated cleanup cell prints the
    full structured report and is the authoritative cleanup entry point.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Section 2 Rule 4, the setup cell must check for
    leftover state with the lab's tag and surface the cleanup command before
    creating any new resources. This prevents the duplicate-Redshift-cluster
    failure mode that the spec calls out.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print("BLOCKED: existing resources tagged labrun:lab-slug=" + LAB_TAG_VALUE + " were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources and")
    print("double the bill. Run the cleanup cell at the bottom of this notebook")
    print("first, or remove them manually with the AWS CLI commands printed by")
    print("the cleanup cell, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Create the staging bucket and upload the payment processor CSVs

The analytics team needs a single bucket where all three processor exports land under a `raw/` prefix. The crawler will point at that prefix later, so every CSV must be inside `raw/` or the crawler will not find it.

Three files, 150 rows each, 450 rows total. Each processor has its own quirks because that is how real vendor exports work:

- **processor_alpha.csv** (150 rows): clean header row, but `created_at` uses a compact ISO 8601 format with no separators (e.g., `20241101T000000Z` instead of `2024-11-01T00:00:00Z`).
- **processor_beta.csv** (150 rows): clean header row, ISO 8601 timestamps in `created_at` (e.g., `2024-11-03T14:22:07Z`).
- **processor_gamma.csv** (150 rows): clean header row, but `created_at` is Unix epoch seconds (e.g., `1730646127`).

Tag the bucket with `labrun:lab-slug=s3-raw-ingestion` at creation time.


In [ ]:
# NBVAL_SKIP
# Task 1: Create S3 bucket, generate 3 sample CSVs, upload to raw/ prefix.
# Tag the bucket with labrun:lab-slug=s3-raw-ingestion at creation time.

import csv
import io
import random

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    region_name=REGION,
)

# --- Create bucket ---
# YOUR CODE: Create the S3 bucket using s3.create_bucket(Bucket=BUCKET_NAME).
# Outside us-east-1 you must also pass CreateBucketConfiguration={"LocationConstraint": REGION}.

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={
        "TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]
    },
)

print(f"Bucket created: {BUCKET_NAME}")

# --- Generate deterministic sample CSVs ---
EXPECTED_TOTAL_ROWS = 450
ROWS_PER_FILE = 150

random.seed(42)  # deterministic so row counts are known at build time

COLUMNS = ["order_id", "customer_id", "amount_cents", "currency", "processor_name", "created_at"]
CURRENCIES = ["USD", "USD", "USD", "EUR", "GBP"]  # weighted toward USD


def _make_rows(processor_name, timestamp_fn, count=ROWS_PER_FILE):
    rows = []
    for i in range(count):
        rows.append([
            f"ord_{processor_name[:5]}_{i:04d}",
            f"cust_{random.randint(1000, 9999)}",
            random.randint(100, 99999),
            random.choice(CURRENCIES),
            processor_name,
            timestamp_fn(i),
        ])
    return rows


def _iso_ts(i):
    base = 1730457600  # 2024-11-01T00:00:00Z
    t = base + i * 300
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(t))


def _compact_iso_ts(i):
    base = 1730457600  # 2024-11-01T00:00:00Z
    t = base + i * 300
    return time.strftime("%Y%m%dT%H%M%SZ", time.gmtime(t))


def _epoch_ts(i):
    base = 1730457600
    return str(base + i * 300)


# processor_alpha: clean header, compact ISO timestamps
alpha_rows = _make_rows("processor_alpha", _compact_iso_ts)
buf_alpha = io.StringIO()
writer = csv.writer(buf_alpha)
writer.writerow(COLUMNS)
for row in alpha_rows:
    writer.writerow(row)
alpha_csv = buf_alpha.getvalue()

# processor_beta: clean header, ISO timestamps
beta_rows = _make_rows("processor_beta", _iso_ts)
buf_beta = io.StringIO()
writer = csv.writer(buf_beta)
writer.writerow(COLUMNS)
for row in beta_rows:
    writer.writerow(row)
beta_csv = buf_beta.getvalue()

# processor_gamma: clean header, epoch timestamps
gamma_rows = _make_rows("processor_gamma", _epoch_ts)
buf_gamma = io.StringIO()
writer = csv.writer(buf_gamma)
writer.writerow(COLUMNS)
for row in gamma_rows:
    writer.writerow(row)
gamma_csv = buf_gamma.getvalue()

# --- Upload to raw/ prefix ---
# YOUR CODE: Upload alpha_csv, beta_csv, and gamma_csv to the raw/ prefix
# using s3.put_object(Bucket, Key, Body). The keys must be raw/processor_alpha.csv,
# raw/processor_beta.csv, and raw/processor_gamma.csv. Encode each CSV string
# as UTF-8 before sending (e.g. alpha_csv.encode("utf-8")).

print(f"\n3 files uploaded to s3://{BUCKET_NAME}/raw/")
print(f"Total rows across all files: {EXPECTED_TOTAL_ROWS}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Bucket exists and contains exactly 3 objects under raw/.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            region_name=REGION,
        )
        # Check bucket exists
        s3_client.head_bucket(Bucket=BUCKET_NAME)

        # Count objects under raw/
        response = s3_client.list_objects_v2(
            Bucket=BUCKET_NAME, Prefix="raw/", MaxKeys=100
        )
        objects = response.get("Contents", [])
        csv_objects = [o for o in objects if o["Key"].endswith(".csv")]

        if len(csv_objects) == 3:
            return CheckpointResult(status="pass")
        else:
            return CheckpointResult(
                status="fail",
                error_reason=f"Expected 3 CSV files under raw/, found {len(csv_objects)}.",
            )
    except ClientError as e:
        error_code = e.response["Error"]["Code"]
        if error_code in ("404", "NoSuchBucket"):
            return CheckpointResult(
                status="fail",
                error_reason=f"Bucket {BUCKET_NAME} does not exist.",
            )
        return CheckpointResult(status="error", error_reason=str(e))
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


In [ ]:
# Wallet check after checkpoint 1.
print("Wallet check: $0.00 so far. S3 storage for a handful of CSVs is free tier.")
print("The crawler in Task 2 is where the meter starts running.")


<details><summary>Hint 1 (nudge)</summary>

The checkpoint says your bucket has zero CSV files under `raw/`. Most likely the upload succeeded but the files landed at the wrong prefix, or the bucket name does not match what the checkpoint is looking for.

</details>

<details><summary>Hint 2 (direction)</summary>

Run `s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="raw/")` and inspect the `Contents` list. If it is empty, your `put_object` calls probably used a key without the `raw/` prefix. The key must start with `raw/` (e.g., `raw/processor_alpha.csv`).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
import csv
import io
import random

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    region_name=REGION,
)

# --- Create bucket ---
if REGION == "us-east-1":
    s3.create_bucket(Bucket=BUCKET_NAME)
else:
    s3.create_bucket(
        Bucket=BUCKET_NAME,
        CreateBucketConfiguration={"LocationConstraint": REGION},
    )

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={
        "TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]
    },
)

print(f"Bucket created: {BUCKET_NAME}")

# --- Generate deterministic sample CSVs ---
EXPECTED_TOTAL_ROWS = 450
ROWS_PER_FILE = 150

random.seed(42)

COLUMNS = ["order_id", "customer_id", "amount_cents", "currency", "processor_name", "created_at"]
CURRENCIES = ["USD", "USD", "USD", "EUR", "GBP"]


def _make_rows(processor_name, timestamp_fn, count=ROWS_PER_FILE):
    rows = []
    for i in range(count):
        rows.append([
            f"ord_{processor_name[:5]}_{i:04d}",
            f"cust_{random.randint(1000, 9999)}",
            random.randint(100, 99999),
            random.choice(CURRENCIES),
            processor_name,
            timestamp_fn(i),
        ])
    return rows


def _iso_ts(i):
    base = 1730457600
    t = base + i * 300
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(t))


def _compact_iso_ts(i):
    base = 1730457600
    t = base + i * 300
    return time.strftime("%Y%m%dT%H%M%SZ", time.gmtime(t))


def _epoch_ts(i):
    base = 1730457600
    return str(base + i * 300)


alpha_rows = _make_rows("processor_alpha", _compact_iso_ts)
buf_alpha = io.StringIO()
writer = csv.writer(buf_alpha)
writer.writerow(COLUMNS)
for row in alpha_rows:
    writer.writerow(row)
alpha_csv = buf_alpha.getvalue()

beta_rows = _make_rows("processor_beta", _iso_ts)
buf_beta = io.StringIO()
writer = csv.writer(buf_beta)
writer.writerow(COLUMNS)
for row in beta_rows:
    writer.writerow(row)
beta_csv = buf_beta.getvalue()

gamma_rows = _make_rows("processor_gamma", _epoch_ts)
buf_gamma = io.StringIO()
writer = csv.writer(buf_gamma)
writer.writerow(COLUMNS)
for row in gamma_rows:
    writer.writerow(row)
gamma_csv = buf_gamma.getvalue()

for filename, body in [
    ("raw/processor_alpha.csv", alpha_csv),
    ("raw/processor_beta.csv", beta_csv),
    ("raw/processor_gamma.csv", gamma_csv),
]:
    s3.put_object(Bucket=BUCKET_NAME, Key=filename, Body=body.encode("utf-8"))
    print(f"Uploaded {filename} ({len(body)} bytes)")

print(f"\n3 files uploaded to s3://{BUCKET_NAME}/raw/")
print(f"Total rows across all files: {EXPECTED_TOTAL_ROWS}")
```

</details>

## Task 2: Create the IAM role, Glue database, and crawler, then run the crawler

The analytics team cannot query anything until the schema is in the Data Catalog. A Glue crawler pointed at the `raw/` prefix will discover the table structure automatically, but the crawler needs an IAM role that can read from the bucket and write to the catalog.

Build these in order:
1. An IAM role with the `AWSGlueServiceRole` managed policy attached, plus an inline policy scoped to your lab bucket (both `s3:GetObject` on objects and `s3:ListBucket` on the bucket itself).
2. A Glue database to hold the discovered table.
3. A Glue crawler pointing at `s3://<your-bucket>/raw/` with the role and database from above.
4. Start the crawler and wait for it to finish.

Tag the IAM role, Glue database, and crawler with `labrun:lab-slug=s3-raw-ingestion`.

The crawler typically takes 2 to 5 minutes to finish on the first run. The polling loop below checks every 15 seconds. Let it run.


In [ ]:
# NBVAL_SKIP
# Task 2: IAM role + Glue database + crawler + run crawler.

import json

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    region_name=REGION,
)

# --- 1. IAM role for the crawler ---
GLUE_TRUST_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "glue.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
})

try:
    # YOUR CODE: Create the IAM role using iam.create_role() with RoleName,
    # AssumeRolePolicyDocument, Description, and Tags. Use GLUE_ROLE_NAME and
    # the GLUE_TRUST_POLICY just defined. Tags format:
    # [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
    print(f"Created IAM role: {GLUE_ROLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"IAM role {GLUE_ROLE_NAME} already exists, reusing.")
    else:
        raise

# Attach the managed policy
MANAGED_POLICY_ARN = "arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole"
iam.attach_role_policy(RoleName=GLUE_ROLE_NAME, PolicyArn=MANAGED_POLICY_ARN)
print("Attached AWSGlueServiceRole managed policy.")

# Inline policy scoped to this bucket only
INLINE_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/raw/*",
        },
        {
            "Effect": "Allow",
            "Action": ["s3:ListBucket"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
        },
    ],
})
# YOUR CODE: Attach INLINE_POLICY to the role using iam.put_role_policy()
# with RoleName, PolicyName (use "labrun-s3-raw-ingestion-s3-access"), and
# PolicyDocument.
print("Attached inline S3 access policy scoped to lab bucket.")

# IAM propagation delay
print("\nYour IAM role is stuck in traffic, give it 10 seconds...")
time.sleep(10)
print("IAM role should be propagated now.")

# --- 2. Glue database ---
try:
    # YOUR CODE: Create the Glue database using glue.create_database() with a
    # DatabaseInput dict containing Name (use GLUE_DATABASE) and Description.
    print(f"\nCreated Glue database: {GLUE_DATABASE}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue database {GLUE_DATABASE} already exists, reusing.")
    else:
        raise

# Tag the database (Glue tagging uses ARN format)
glue.tag_resource(
    ResourceArn=f"arn:aws:glue:{REGION}:{account_id}:database/{GLUE_DATABASE}",
    TagsToAdd={LAB_TAG_KEY: LAB_TAG_VALUE},
)

# --- 3. Create the crawler ---
ROLE_ARN = f"arn:aws:iam::{account_id}:role/{GLUE_ROLE_NAME}"

try:
    # YOUR CODE: Create the Glue crawler using glue.create_crawler() with Name
    # (use GLUE_CRAWLER), Role (use ROLE_ARN), DatabaseName, Targets, SchemaChangePolicy,
    # and Tags. Targets format: {"S3Targets": [{"Path": f"s3://{BUCKET_NAME}/raw/"}]}.
    # SchemaChangePolicy format: {"UpdateBehavior": "UPDATE_IN_DATABASE", "DeleteBehavior": "LOG"}.
    print(f"Created Glue crawler: {GLUE_CRAWLER}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Crawler {GLUE_CRAWLER} already exists, reusing.")
    else:
        raise

# --- 4. Run the crawler ---
# YOUR CODE: Start the crawler using glue.start_crawler(Name=GLUE_CRAWLER).
print(f"\nStarted crawler {GLUE_CRAWLER}. Waiting for it to finish...")

while True:
    resp = glue.get_crawler(Name=GLUE_CRAWLER)
    state = resp["Crawler"]["State"]
    if state == "READY":
        last_crawl = resp["Crawler"].get("LastCrawl", {})
        status = last_crawl.get("Status", "UNKNOWN")
        print(f"Crawler finished with status: {status}")
        if status == "SUCCEEDED":
            tables_resp = glue.get_tables(DatabaseName=GLUE_DATABASE)
            tables = tables_resp.get("TableList", [])
            print(f"Tables in catalog: {len(tables)}")
            for t in tables:
                col_count = len(t["StorageDescriptor"]["Columns"])
                print(f"  {t['Name']} ({col_count} columns)")
        break
    print("Poking your Glue crawler to see if it woke up...")
    time.sleep(15)


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Crawler exists, ran successfully, created at least one table.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            region_name=REGION,
        )
        resp = glue_client.get_crawler(Name=GLUE_CRAWLER)
        crawler = resp["Crawler"]
        last_crawl = crawler.get("LastCrawl", {})
        crawl_status = last_crawl.get("Status", "UNKNOWN")

        if crawl_status != "SUCCEEDED":
            return CheckpointResult(
                status="fail",
                error_reason=f"Crawler last run status is '{crawl_status}', expected 'SUCCEEDED'.",
            )

        # Check at least one table exists in the database
        tables_resp = glue_client.get_tables(DatabaseName=GLUE_DATABASE)
        table_count = len(tables_resp.get("TableList", []))
        if table_count == 0:
            return CheckpointResult(
                status="fail",
                error_reason="Crawler succeeded but created zero tables in the Data Catalog.",
            )

        return CheckpointResult(status="pass")
    except ClientError as e:
        error_code = e.response["Error"]["Code"]
        if error_code == "EntityNotFoundException":
            return CheckpointResult(
                status="fail",
                error_reason=f"Crawler {GLUE_CRAWLER} does not exist.",
            )
        return CheckpointResult(status="error", error_reason=str(e))
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


In [ ]:
# Wallet check after checkpoint 2.
print("Wallet check: 1 crawler run so far ($0.015), total damage: about 1.5 cents.")
print("If you had to re-run the crawler after fixing IAM, add another 1.5 cents per attempt.")


<details><summary>Hint 1 (nudge)</summary>

The crawler ran but the Data Catalog shows zero tables. The crawler can succeed at the AWS level and still discover nothing. Most often it is a permissions or path issue.

</details>

<details><summary>Hint 2 (direction)</summary>

Run `s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="raw/")` and confirm at least one `.csv` file is listed. If the files are there, the crawler's IAM role probably cannot read them. Check that the inline policy includes both `s3:GetObject` on the objects AND `s3:ListBucket` on the bucket itself.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
import json

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    region_name=REGION,
)

# --- 1. IAM role for the crawler ---
GLUE_TRUST_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "glue.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
})

try:
    iam.create_role(
        RoleName=GLUE_ROLE_NAME,
        AssumeRolePolicyDocument=GLUE_TRUST_POLICY,
        Description="Glue crawler role for labrun s3-raw-ingestion lab",
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Created IAM role: {GLUE_ROLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"IAM role {GLUE_ROLE_NAME} already exists, reusing.")
    else:
        raise

MANAGED_POLICY_ARN = "arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole"
iam.attach_role_policy(RoleName=GLUE_ROLE_NAME, PolicyArn=MANAGED_POLICY_ARN)
print("Attached AWSGlueServiceRole managed policy.")

INLINE_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/raw/*",
        },
        {
            "Effect": "Allow",
            "Action": ["s3:ListBucket"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
        },
    ],
})
iam.put_role_policy(
    RoleName=GLUE_ROLE_NAME,
    PolicyName="labrun-s3-raw-ingestion-s3-access",
    PolicyDocument=INLINE_POLICY,
)
print("Attached inline S3 access policy scoped to lab bucket.")

print("\nYour IAM role is stuck in traffic, give it 10 seconds...")
time.sleep(10)
print("IAM role should be propagated now.")

# --- 2. Glue database ---
try:
    glue.create_database(
        DatabaseInput={
            "Name": GLUE_DATABASE,
            "Description": "labrun s3-raw-ingestion lab database",
        },
    )
    print(f"\nCreated Glue database: {GLUE_DATABASE}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue database {GLUE_DATABASE} already exists, reusing.")
    else:
        raise

glue.tag_resource(
    ResourceArn=f"arn:aws:glue:{REGION}:{account_id}:database/{GLUE_DATABASE}",
    TagsToAdd={LAB_TAG_KEY: LAB_TAG_VALUE},
)

# --- 3. Create the crawler ---
ROLE_ARN = f"arn:aws:iam::{account_id}:role/{GLUE_ROLE_NAME}"

try:
    glue.create_crawler(
        Name=GLUE_CRAWLER,
        Role=ROLE_ARN,
        DatabaseName=GLUE_DATABASE,
        Targets={"S3Targets": [{"Path": f"s3://{BUCKET_NAME}/raw/"}]},
        SchemaChangePolicy={
            "UpdateBehavior": "UPDATE_IN_DATABASE",
            "DeleteBehavior": "LOG",
        },
        Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
    print(f"Created Glue crawler: {GLUE_CRAWLER}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Crawler {GLUE_CRAWLER} already exists, reusing.")
    else:
        raise

# --- 4. Run the crawler ---
glue.start_crawler(Name=GLUE_CRAWLER)
print(f"\nStarted crawler {GLUE_CRAWLER}. Waiting for it to finish...")

while True:
    resp = glue.get_crawler(Name=GLUE_CRAWLER)
    state = resp["Crawler"]["State"]
    if state == "READY":
        last_crawl = resp["Crawler"].get("LastCrawl", {})
        status = last_crawl.get("Status", "UNKNOWN")
        print(f"Crawler finished with status: {status}")
        if status == "SUCCEEDED":
            tables_resp = glue.get_tables(DatabaseName=GLUE_DATABASE)
            tables = tables_resp.get("TableList", [])
            print(f"Tables in catalog: {len(tables)}")
            for t in tables:
                col_count = len(t["StorageDescriptor"]["Columns"])
                print(f"  {t['Name']} ({col_count} columns)")
        break
    print("Poking your Glue crawler to see if it woke up...")
    time.sleep(15)
```

</details>

## Task 3: Verify the Glue Data Catalog table schema

The crawler finished and the Data Catalog now has a table, but the analytics team will not trust it until someone confirms the column names and types actually match the source CSVs.

Inspect the table the crawler created. The columns should match the expected schema (`order_id`, `customer_id`, `amount_cents`, `currency`, `processor_name`, `created_at`), and numeric columns like `amount_cents` should land as `bigint`. Expect `created_at` to come back as `string` even though it carries timestamps: the three processors use three different formats (compact ISO 8601 in alpha, standard ISO 8601 in beta, Unix epoch seconds in gamma), and when the crawler sees incompatible representations in the same column it falls back to the safest type.


In [ ]:
# NBVAL_SKIP
# Task 3: Inspect the Data Catalog table schema.

# YOUR CODE: Fetch all tables from the Glue Data Catalog database using
# glue.get_tables(DatabaseName=GLUE_DATABASE) and store the response in a
# variable called tables_resp.

tables = tables_resp.get("TableList", [])

if not tables:
    print("ERROR: No tables found in the Data Catalog database.")
    print(f"Database: {GLUE_DATABASE}")
    print("The crawler may not have discovered any data. Check the hints below.")
else:
    for table in tables:
        table_name = table["Name"]
        print(f"Table: {GLUE_DATABASE}.{table_name}")
        print(f"Location: {table['StorageDescriptor']['Location']}")
        print(f"Input format: {table['StorageDescriptor'].get('InputFormat', 'N/A')}")
        print()
        print("Columns:")
        print(f"  {'Name':<20} {'Type':<15}")
        print(f"  {'-' * 20} {'-' * 15}")
        for col in table["StorageDescriptor"]["Columns"]:
            print(f"  {col['Name']:<20} {col['Type']:<15}")

        print()
        print("Partition keys:", [p["Name"] for p in table.get("PartitionKeys", [])] or "none")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Table exists in catalog with expected columns.

EXPECTED_COLUMNS = {"order_id", "customer_id", "amount_cents", "currency", "processor_name", "created_at"}


def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            region_name=REGION,
        )
        tables_resp = glue_client.get_tables(DatabaseName=GLUE_DATABASE)
        tables = tables_resp.get("TableList", [])

        if not tables:
            return CheckpointResult(
                status="fail",
                error_reason="No tables found in the Data Catalog database.",
            )

        table = tables[0]
        columns = table["StorageDescriptor"]["Columns"]
        col_names = {c["Name"] for c in columns}
        col_types = {c["Type"] for c in columns}

        # Check that expected columns are present
        missing = EXPECTED_COLUMNS - col_names
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=f"Missing expected columns: {', '.join(sorted(missing))}. Found: {', '.join(sorted(col_names))}",
            )

        # Check that not every column is string (crawler should infer some types)
        if col_types == {"string"}:
            return CheckpointResult(
                status="fail",
                error_reason="Every column is typed as 'string'. The crawler could not infer numeric or timestamp types. Check the hints for common causes.",
            )

        return CheckpointResult(status="pass")
    except ClientError as e:
        error_code = e.response["Error"]["Code"]
        if error_code == "EntityNotFoundException":
            return CheckpointResult(
                status="fail",
                error_reason=f"Database {GLUE_DATABASE} does not exist.",
            )
        return CheckpointResult(status="error", error_reason=str(e))
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


In [ ]:
# Wallet check after checkpoint 3.
print("Wallet check: still at about 1.5 cents (1 crawler run).")
print("Schema inspection calls to the Data Catalog are free.")


<details><summary>Hint 1 (nudge)</summary>

The crawler created a table but inferred every column as `string`, including the amount and timestamp columns. This is the default when the classifier cannot reconcile the formats it sees across files in the same prefix.

</details>

<details><summary>Hint 2 (direction)</summary>

The crawler inferred column types from your CSVs, but mixed formats in the same column can confuse the classifier. Compare what the crawler detected vs what the data actually contains. Look at `created_at` across `processor_beta.csv` (ISO 8601 timestamps) and `processor_gamma.csv` (Unix epoch seconds). When the crawler sees two incompatible representations in the same column, it falls back to the safest type.

</details>

<details><summary>Hint 3 (spoiler)</summary>

The root cause is the three files using three different `created_at` formats: compact ISO 8601 in `processor_alpha.csv`, standard ISO 8601 in `processor_beta.csv`, and Unix epoch seconds in `processor_gamma.csv`. The crawler merges schemas across files and falls back to `string` for any column where it cannot pick one type. After the crawler runs, the table should still appear with `bigint` for `amount_cents` and `string` for the other columns.

Here is the complete working solution for Task 3:

```python
tables_resp = glue.get_tables(DatabaseName=GLUE_DATABASE)
tables = tables_resp.get("TableList", [])

if not tables:
    print("ERROR: No tables found in the Data Catalog database.")
    print(f"Database: {GLUE_DATABASE}")
    print("The crawler may not have discovered any data. Check the hints below.")
else:
    for table in tables:
        table_name = table["Name"]
        print(f"Table: {GLUE_DATABASE}.{table_name}")
        print(f"Location: {table['StorageDescriptor']['Location']}")
        print(f"Input format: {table['StorageDescriptor'].get('InputFormat', 'N/A')}")
        print()
        print("Columns:")
        print(f"  {'Name':<20} {'Type':<15}")
        print(f"  {'-' * 20} {'-' * 15}")
        for col in table["StorageDescriptor"]["Columns"]:
            print(f"  {col['Name']:<20} {col['Type']:<15}")

        print()
        print("Partition keys:", [p["Name"] for p in table.get("PartitionKeys", [])] or "none")
```

</details>

## Task 4: Run an Athena query to validate the row count

The schema looks right, but the analytics team wants one more proof point before they trust this pipeline: does the table contain exactly the same number of rows as the source files? A `SELECT COUNT(*)` from Athena against the catalog table should return 450 (150 rows per processor file).

Before you can query, Athena needs a results location. Point it at `s3://<your-bucket>/athena-results/` so the query output stays inside the lab bucket and gets cleaned up with everything else.


In [ ]:
# NBVAL_SKIP
# Task 4: Configure Athena results location, run COUNT(*), validate row count.

athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    region_name=REGION,
)

# Get the table name from the catalog (crawler names it based on the S3 prefix)
tables_resp = glue.get_tables(DatabaseName=GLUE_DATABASE)
TABLE_NAME = tables_resp["TableList"][0]["Name"]
print(f"Querying table: {GLUE_DATABASE}.{TABLE_NAME}")

ATHENA_OUTPUT = f"s3://{BUCKET_NAME}/athena-results/"

# Run the COUNT(*) query
query = f'SELECT COUNT(*) AS total_rows FROM "{GLUE_DATABASE}"."{TABLE_NAME}"'
print(f"Running: {query}")

# YOUR CODE: Start the Athena query using athena.start_query_execution() with
# QueryString=query and ResultConfiguration={"OutputLocation": ATHENA_OUTPUT}.
# Save the response to a variable called start_resp.
query_id = start_resp["QueryExecutionId"]

# Poll for completion
print("Asking Athena to put on her reading glasses...")
while True:
    status_resp = athena.get_query_execution(QueryExecutionId=query_id)
    state = status_resp["QueryExecution"]["Status"]["State"]
    if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
        break
    print("Athena is still rummaging through your data lake...")
    time.sleep(3)

if state != "SUCCEEDED":
    reason = status_resp["QueryExecution"]["Status"].get("StateChangeReason", "unknown")
    print(f"ERROR: Query {state}. Reason: {reason}")
else:
    # YOUR CODE: Fetch the query results using athena.get_query_results() with
    # QueryExecutionId=query_id. Save the response to a variable called result_resp.
    rows = result_resp["ResultSet"]["Rows"]
    # Row 0 is the header, row 1 is the data
    actual_count = int(rows[1]["Data"][0]["VarCharValue"])
    print(f"\nAthena row count: {actual_count}")
    print(f"Expected row count: {EXPECTED_TOTAL_ROWS}")
    if actual_count == EXPECTED_TOTAL_ROWS:
        print("Row counts match. The pipeline delivered every row.")
    else:
        print(f"MISMATCH: expected {EXPECTED_TOTAL_ROWS}, got {actual_count}.")
        print("Check the hints below for common causes.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Athena COUNT(*) equals the expected total rows.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        athena_client = boto3.client(
            "athena",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            region_name=REGION,
        )
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            region_name=REGION,
        )

        # Get table name
        tables_resp = glue_client.get_tables(DatabaseName=GLUE_DATABASE)
        tables = tables_resp.get("TableList", [])
        if not tables:
            return CheckpointResult(
                status="fail",
                error_reason="No tables in the Data Catalog to query.",
            )
        table_name = tables[0]["Name"]

        # Run COUNT(*)
        query = f'SELECT COUNT(*) AS total_rows FROM "{GLUE_DATABASE}"."{table_name}"'
        start_resp = athena_client.start_query_execution(
            QueryString=query,
            ResultConfiguration={
                "OutputLocation": f"s3://{BUCKET_NAME}/athena-results/"
            },
        )
        query_id = start_resp["QueryExecutionId"]

        # Poll (max 60 seconds)
        for _ in range(20):
            status_resp = athena_client.get_query_execution(QueryExecutionId=query_id)
            state = status_resp["QueryExecution"]["Status"]["State"]
            if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
                break
            time.sleep(3)

        if state != "SUCCEEDED":
            reason = status_resp["QueryExecution"]["Status"].get(
                "StateChangeReason", "unknown"
            )
            return CheckpointResult(
                status="error",
                error_reason=f"Athena query {state}: {reason}",
            )

        result_resp = athena_client.get_query_results(QueryExecutionId=query_id)
        rows = result_resp["ResultSet"]["Rows"]
        actual_count = int(rows[1]["Data"][0]["VarCharValue"])

        if actual_count == EXPECTED_TOTAL_ROWS:
            return CheckpointResult(status="pass")
        else:
            return CheckpointResult(
                status="fail",
                error_reason=f"Row count mismatch: expected {EXPECTED_TOTAL_ROWS}, got {actual_count}.",
            )
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


In [ ]:
# Wallet check after checkpoint 4.
print("Wallet check: 1 crawler run ($0.015) + 1 Athena query (fraction of a penny).")
print("Total damage so far: about 2 cents. Your morning coffee cost 200x more.")


<details><summary>Hint 1 (nudge)</summary>

Athena returned an error about query results location, or the COUNT came back as zero. Both are fixable in under two minutes. The first is a workgroup setting, the second is usually a path issue in the Data Catalog table.

</details>

<details><summary>Hint 2 (direction)</summary>

For the results location error, make sure you pass `ResultConfiguration` with an `OutputLocation` pointing at your bucket:

```python
athena.start_query_execution(
    QueryString=query,
    ResultConfiguration={"OutputLocation": f"s3://{BUCKET_NAME}/athena-results/"},
)
```

For a zero COUNT, inspect the table's storage location:

```python
table = glue.get_table(DatabaseName=GLUE_DATABASE, Name=TABLE_NAME)
print(table["Table"]["StorageDescriptor"]["Location"])
```

It should point at `s3://<your-bucket>/raw/`. If it points somewhere else, the crawler used the wrong target path.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    region_name=REGION,
)

tables_resp = glue.get_tables(DatabaseName=GLUE_DATABASE)
TABLE_NAME = tables_resp["TableList"][0]["Name"]
print(f"Querying table: {GLUE_DATABASE}.{TABLE_NAME}")

ATHENA_OUTPUT = f"s3://{BUCKET_NAME}/athena-results/"

query = f'SELECT COUNT(*) AS total_rows FROM "{GLUE_DATABASE}"."{TABLE_NAME}"'
print(f"Running: {query}")

start_resp = athena.start_query_execution(
    QueryString=query,
    ResultConfiguration={"OutputLocation": ATHENA_OUTPUT},
)
query_id = start_resp["QueryExecutionId"]

print("Asking Athena to put on her reading glasses...")
while True:
    status_resp = athena.get_query_execution(QueryExecutionId=query_id)
    state = status_resp["QueryExecution"]["Status"]["State"]
    if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
        break
    print("Athena is still rummaging through your data lake...")
    time.sleep(3)

if state != "SUCCEEDED":
    reason = status_resp["QueryExecution"]["Status"].get("StateChangeReason", "unknown")
    print(f"ERROR: Query {state}. Reason: {reason}")
else:
    result_resp = athena.get_query_results(QueryExecutionId=query_id)
    rows = result_resp["ResultSet"]["Rows"]
    actual_count = int(rows[1]["Data"][0]["VarCharValue"])
    print(f"\nAthena row count: {actual_count}")
    print(f"Expected row count: {EXPECTED_TOTAL_ROWS}")
    if actual_count == EXPECTED_TOTAL_ROWS:
        print("Row counts match. The pipeline delivered every row.")
    else:
        print(f"MISMATCH: expected {EXPECTED_TOTAL_ROWS}, got {actual_count}.")
        print("Check the hints below for common causes.")
```

The expected row count is **450** (150 rows in each of the 3 processor CSV files). If you get a different number, one of the files may not have uploaded correctly, or the crawler excluded it from the table.

</details>

## Cleanup

Time to tear it all down. The cell below runs through your manifest in reverse order, deletes every resource, then double-checks AWS to confirm nothing is left running. If anything failed, you get the exact CLI command to finish the job manually. Re-running is safe.


In [ ]:
# NBVAL_SKIP
# Cleanup. This cell calls run_cleanup against the manifest, prints the
# structured warnings and orphans, then posts the cleanup status to the
# Labrun API via cleanup(). The notebook never implements teardown or
# verification logic itself.

result = run_cleanup(CLEANUP_MANIFEST)

print("=" * 60)
print("Cleanup report for lab:", LAB_ID)
print("=" * 60)
print("Status:", result.status)
print()

if result.warnings:
    print("Warnings (resources that did not delete cleanly):")
    for w in result.warnings:
        print("  -", w)
    print()
else:
    print("No deletion warnings.")
    print()

if result.orphans:
    print("Orphans (tagged resources not in this session's manifest):")
    for o in result.orphans:
        print("  -", o)
    print()
else:
    print("No tag-audit orphans.")
    print()

print("=" * 60)
if result.status == "clean":
    print("All clear. Every resource this lab created has been verified")
    print("deleted. Open your AWS Billing console in 24 hours to confirm:")
    print("https://console.aws.amazon.com/billing/")
else:
    print("Cleanup is dirty. Run the CLI commands listed above to remove")
    print("the leftovers, then re-run this cell. Your AWS account may")
    print("still be accruing charges until the warnings clear.")
print("=" * 60)

# Canonical summary per RESOURCE-SAFETY-SPEC Rule 6.
total_resources = len(CLEANUP_MANIFEST)
critical_count = 0  # Lab 01 has no hourly-billed resources.
failed_count = sum(
    1 for w in result.warnings
    if w.startswith("FAILED TO DELETE:") or "is still alive" in w
)
standard_count = total_resources - critical_count - failed_count

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_count}")
print(f"Standard resources destroyed: {standard_count}")
print(f"Resources that failed to delete: {failed_count}")

cleanup(status=result.status)

if failed_count > 0:
    raise SystemExit(1)


## Session total

Approximate session cost: about 2 cents on a clean first run (1 crawler run at \$0.015, 1 Athena query at a fraction of a penny). If you re-ran the crawler once or twice while debugging the IAM policy, add 1.5 cents per extra run. A realistic total with some debugging is 5 to 15 cents.

Check your AWS Billing console in 24 hours to see the exact number. If you ran cleanup and the report above shows status `clean` with zero warnings and zero orphans, your bill should match this estimate closely.


## Reflection

These are not graded. They are for you.

1. The crawler merged three CSV files into one table, but `created_at` came back typed as `string` even though every row carries a timestamp. Why did the crawler fall back to `string`, and what would change if all three processors switched to the same timestamp format?

2. What would break if the payment-processor team added a fourth CSV tomorrow with a new column called `refund_status`? Would the existing crawler handle it, or would the Data Catalog table need manual intervention?

3. The inline IAM policy you wrote scoped `s3:GetObject` to `raw/*` and `s3:ListBucket` to the bucket itself. Why does the crawler need both permissions, and what specific error would you see if you removed `s3:ListBucket`?

## Exam-Style Practice

**Q1.** You configured the Glue crawler to point at `s3://bucket/raw/` rather than `s3://bucket/`. A teammate suggests pointing crawlers at the bucket root to "discover everything." What is the trade-off of crawling the bucket root in a multi-team data lake?

A) The crawler runs faster because it batches all prefixes in a single API call.

B) The crawler may merge schemas from unrelated datasets and produce a confusing single table.

C) Athena automatically rejects queries against tables created from bucket-root crawls.

D) Glue refuses to create tables when more than one prefix is present under the target.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because crawler runtime scales with object count, not API batching; pointing at the root usually makes the crawler slower, not faster.
- B) Right because Glue groups files into tables by schema similarity. Heterogeneous data under one root can collapse into a single table with every column typed as `string`, the same fallback you saw on `created_at` in this lab.
- C) Wrong because Athena queries any catalog table regardless of how the crawler was scoped.
- D) Wrong because Glue happily produces multiple tables under one prefix; the limit is logical (schema confusion), not enforced by the API.

</details>

**Q2.** Three CSV files share an S3 prefix and a column called `event_time`. File A uses ISO 8601 strings (`2024-11-01T00:00:00Z`), File B uses Unix epoch seconds (`1730457600`), and File C uses compact ISO 8601 (`20241101T000000Z`). When the Glue crawler runs against this prefix, what type does it assign to `event_time`?

A) `timestamp` because all three formats represent timestamps.

B) `bigint` because epoch seconds are numeric and the crawler picks the most common type.

C) `string` because the crawler cannot reconcile incompatible formats and falls back to the safest type.

D) The crawler creates three separate tables, one per format.

<details><summary>Show answer</summary>

**C**.

- A) Wrong because the crawler can detect ISO 8601 timestamps only when every file in the prefix uses the same format; mixed formats break inference.
- B) Wrong because the crawler does not "vote" on type by file count; one non-numeric value in a column blocks a numeric result.
- C) Right because this is exactly what happened with `created_at` in this lab. The crawler merges schemas across files in the same prefix and falls back to `string` for any column whose formats are incompatible.
- D) Wrong because the crawler groups files by schema similarity, not by format variation in a single column; it produces one table here.

</details>

**Q3.** A team queries an Athena table backed by 100 GB of unpartitioned CSV files in S3. The query is `SELECT COUNT(*) FROM events WHERE created_at >= '2024-11-01'`. Which change reduces both query time and cost the most?

A) Replace `COUNT(*)` with `COUNT(1)` to skip a full table scan.

B) Convert the data to Parquet partitioned by `created_at` date and re-crawl.

C) Add a `LIMIT 1000` clause so Athena reads less data.

D) Pre-compute the count nightly into a static text file in S3 and read that instead.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because `COUNT(1)` and `COUNT(*)` are equivalent in Athena; both still scan the table.
- B) Right because Parquet's columnar layout lets Athena read only the `created_at` column, and date-partitioning lets it prune files outside the filter range. The combination typically cuts scanned bytes by 10-100x and Athena bills on bytes scanned.
- C) Wrong because `LIMIT` is applied after Athena reads the source data; cost is based on scan size, not result rows.
- D) Wrong because static pre-computed counts go stale the moment new data lands and bypass the catalog you just built.

</details>
